# Remoção de artefactos em eletroencefalografia (EEG)

Este notebook acompanha o processo de pré-processamento de um registo de EEG real, desde o sinal não processado até ao sinal livre dos principais artefactos, seguindo os passos habitualmente utilizados em investigação.

**Não são necessários conhecimentos de programação.**
- Cada bloco é executado selecionando-o e premindo **Shift + Enter** (ou o botão ▶). Os blocos devem ser executados **pela ordem apresentada**.
- Os blocos identificados apenas por um título (preparação e funções auxiliares) têm o código recolhido; o código pode ser consultado em **Show code**.
- Os blocos de código visíveis correspondem às **etapas do processamento**. Cada etapa é uma instrução curta, que pode ser reordenada ou adaptada a outros registos.

### Conteúdo
1. **O registo de EEG**: características e disposição dos elétrodos
2. **Identificação de artefactos**: sinais de origem não cerebral
3. **Filtragem**: remoção de componentes de muito baixa e de alta frequência
4. **Elétrodos com mau contacto**: deteção e interpolação
5. **O artefacto ocular**: caracterização do pestanejo
6. **Análise de Componentes Independentes (ICA)**: separação entre atividade cerebral e artefactos
7. **Avaliação do resultado**: comparação antes e depois da correção

Será utilizado o **[MNE-Python](https://mne.tools)**, uma biblioteca de código aberto amplamente utilizada na análise de sinais de EEG e MEG.

In [ ]:
#@title Preparação: instalação e carregamento das bibliotecas
# No Colab, a biblioteca MNE-Python é instalada através do gestor de pacotes pip; o relatório
# detalhado da instalação só é apresentado se ocorrer um erro. Num ambiente Jupyter local,
# assume-se que as bibliotecas já estão instaladas (ver README do projeto).
import subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print("A instalar o MNE...")
    result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "mne"],
                            capture_output=True, text=True)
    print("Instalação concluída." if result.returncode == 0 else result.stderr)

# Bibliotecas utilizadas:
#   numpy      - cálculo numérico com vetores e matrizes
#   matplotlib - produção de gráficos
#   mne        - leitura, processamento e visualização de sinais de EEG
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import mne
from mne.datasets import eegbci

mne.set_log_level("ERROR")                 # o MNE apresenta apenas mensagens de erro
mne.viz.set_browser_backend("matplotlib")  # os sinais são apresentados como imagens estáticas
plt.rcParams["figure.dpi"] = 100           # resolução das figuras

# Pasta onde são guardados os registos descarregados: no Colab, na sessão; localmente, na pasta
# data/ do projeto (e não dentro de notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)


def download_example(file_name):
    # Descarrega um registo da base de dados PhysioNet (apenas na primeira execução) e devolve o
    # caminho do ficheiro. O nome indica o participante e a sessão: "S001R04.edf" = participante 1, sessão 4.
    subject, run = int(file_name[1:4]), int(file_name[5:7])
    return eegbci.load_data(subject, [run], path=RAW_DIR, update_path=False)[0]

## 1. O registo de EEG

O **EEG** regista, através de elétrodos colocados no couro cabeludo, a atividade elétrica cortical. A amplitude destes sinais é da ordem dos **microvolts (µV)**, ou seja, milionésimos de volt, o que os torna particularmente suscetíveis a interferências de outras origens.

O registo utilizado provém de uma base de dados pública ([PhysioNet EEG Motor Movement/Imagery](https://physionet.org/content/eegmmidb/1.0.0/)), que inclui registos de 109 participantes saudáveis durante tarefas motoras simples (abertura e fecho das mãos). O ficheiro é descarregado automaticamente na primeira execução.

Carregamento do registo:

In [ ]:
fname = download_example("S001R04.edf")
raw = mne.io.read_raw_edf(fname, preload=True)

eegbci.standardize(raw)             # uniformizar os nomes dos elétrodos (ex.: "Fp1." -> "Fp1")
raw.set_montage("standard_1005")    # associar cada elétrodo à sua posição no escalpe (sistema 10-10)
raw.set_annotations(None)           # remover as marcações da tarefa motora, não utilizadas aqui

print(f"Número de elétrodos      : {len(raw.ch_names)}")
print(f"Duração do registo       : {raw.times[-1]:.0f} segundos")
print(f"Frequência de amostragem : {raw.info['sfreq']:.0f} Hz")

O registo inclui 64 elétrodos, com uma **frequência de amostragem de 160 Hz** (160 amostras por segundo em cada elétrodo).

A figura seguinte apresenta a disposição dos elétrodos, segundo o sistema internacional 10-10, em vista superior (nariz orientado para cima). A nomenclatura indica a região: **Fp** = frontopolar, **F** = frontal, **C** = central, **T** = temporal, **P** = parietal, **O** = occipital. Os números ímpares correspondem ao hemisfério esquerdo, os pares ao direito, e **z** à linha média. Os elétrodos **Fp1** e **Fp2**, situados imediatamente acima dos olhos, serão relevantes adiante.

In [ ]:
raw.plot_sensors(show_names=True, sphere="eeglab");

In [ ]:
#@title Funções auxiliares de visualização
# Funções utilizadas nas figuras ao longo do notebook. Não alteram os dados.

# Subconjunto de elétrodos apresentado nas figuras, de anterior para posterior
SHOW = ["Fp1", "Fp2", "AF3", "F3", "Fz", "C3", "Cz", "P3", "Pz", "O1", "Oz"]

# Identificar os instantes dos pestanejos (deflexões amplas em Fp1/Fp2, após filtrar 1-15 Hz)
# e selecionar o intervalo de 10 segundos que contém mais pestanejos, para as figuras
blinks = mne.preprocessing.find_eog_events(raw.copy().filter(1, 15), ch_name=["Fp1", "Fp2"])
BLINK_TIMES = blinks[:, 0] / raw.info["sfreq"] - raw.first_time
START = max(np.arange(0, raw.times[-1] - 10, 0.5),
            key=lambda s: np.sum((BLINK_TIMES > s + 1) & (BLINK_TIMES < s + 9)))


def show(recording, title, start=START, duration=10, channels=SHOW, spacing=150, mark_blinks=False):
    # Representa os mesmos elétrodos no mesmo intervalo de tempo, para que as figuras sejam comparáveis.
    # Cada elétrodo é desenhado numa linha própria; mark_blinks=True assinala os pestanejos a azul.
    idx = [recording.ch_names.index(ch) for ch in channels]
    data, times = recording.get_data(picks=idx, tmin=start, tmax=start + duration, return_times=True)
    data = (data - data.mean(axis=1, keepdims=True)) * 1e6   # remover o valor médio e converter V -> µV
    fig, ax = plt.subplots(figsize=(12, 0.45 * len(channels) + 1.2))
    for i, trace in enumerate(data):
        ax.plot(times, trace - i * spacing, color="black", lw=0.7)
    if mark_blinks:
        for t in BLINK_TIMES[(BLINK_TIMES > start) & (BLINK_TIMES < start + duration)]:
            ax.axvspan(t - 0.2, t + 0.2, color="tab:blue", alpha=0.15)
    ax.set_yticks(-np.arange(len(channels)) * spacing, channels)
    ax.set_ylim(-len(channels) * spacing, 1.5 * spacing)
    ax.set_xlim(times[0], times[-1])
    ax.set_xlabel("Tempo (segundos)")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.spines[["top", "right", "left"]].set_visible(False)
    # Barra de escala vertical de 100 µV, como referência de amplitude
    ax.plot([times[-1]] * 2, [spacing * 0.3, spacing * 0.3 + 100], color="tab:red", lw=3, clip_on=False)
    ax.text(times[-1], spacing * 0.3 + 50, "  100 µV", color="tab:red", va="center")
    plt.show()


def compare(before, after, title, channel="Fp1", start=START, duration=10):
    # Sobrepõe o sinal de um elétrodo antes (vermelho) e depois (preto) de um passo de processamento
    fig, ax = plt.subplots(figsize=(12, 3))
    for recording, color, label in [(before, "tab:red", "antes"), (after, "black", "depois")]:
        data, times = recording.get_data(picks=[channel], tmin=start, tmax=start + duration,
                                         return_times=True)
        ax.plot(times, data[0] * 1e6, color=color, lw=0.9, label=label)
    ax.set_xlabel("Tempo (segundos)")
    ax.set_ylabel(f"{channel} (µV)")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.legend(loc="upper right")
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()


def spectrum(recording, title):
    # Densidade espectral de potência (média de todos os elétrodos): a contribuição de cada frequência
    fig = recording.compute_psd(fmax=80).plot(average=True, amplitude=False, show=False)
    ax = fig.axes[0]
    ax.set_title("")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_xlabel("Frequência (Hz)")
    ax.set_ylabel("Potência (dB)")
    plt.show()


def show_average(evoked, title, ylim=None, topomap=False):
    # Representa uma média de segmentos (ex.: o pestanejo médio); topomap=True acrescenta
    # o mapa topográfico no instante 0
    if topomap:
        fig = evoked.plot_joint(title=title, times=[0.0], ts_args=dict(gfp=False), show=False)
    else:
        fig = evoked.plot(ylim=ylim, gfp=False, show=False)
        fig.axes[0].set_title(title)
    for ax in fig.axes:
        if ax.get_xlabel() == "Time (s)":
            ax.set_xlabel("Tempo (s)")
    plt.show()


def show_correlation(ica, scores, components):
    # Correlação de cada componente ICA com os elétrodos frontopolares; o(s) componente(s) indicado(s) a vermelho
    fig = ica.plot_scores(scores, exclude=components, show=False,
                          title="Correlação de cada componente com os elétrodos frontopolares")
    for ax in fig.axes:
        ax.set_xlabel("Componente")
        ax.set_ylabel("Correlação")
    plt.show()


def show_components_over_time(ica, recording, n=5):
    # Evolução temporal dos primeiros n componentes ICA, no mesmo intervalo das restantes figuras
    fig = ica.plot_sources(recording, picks=range(n), start=START, stop=START + 10,
                           show_scrollbars=False, show=False)
    fig.mne.ax_main.set_xlabel("Tempo (s)")
    plt.show()

## 2. O sinal não processado

A figura seguinte apresenta 10 segundos do sinal, num subconjunto de elétrodos ordenados da região frontopolar (**em cima**) para a região occipital (**em baixo**).

In [ ]:
show(raw, "Sinal não processado")

Nos elétrodos **Fp1 e Fp2** (linhas superiores) observam-se deflexões de grande amplitude que se destacam da atividade de fundo. Na figura seguinte, estes eventos encontram-se assinalados:

In [ ]:
show(raw, "Sinal não processado, com os pestanejos assinalados", mark_blinks=True)

Trata-se de **pestanejos**. A sua amplitude é muito superior à da atividade cerebral, mas a sua origem não é cerebral: o globo ocular comporta-se como um dipolo elétrico (a córnea é positiva em relação à retina), e o movimento da pálpebra altera o campo elétrico na região frontal.

Designam-se **artefactos** todos os sinais registados pelos elétrodos que **não têm origem na atividade cerebral**. A sua remoção é necessária para evitar que sejam interpretados como atividade neuronal.

| Artefacto | Origem | Características |
|---|---|---|
| Pestanejo | dipolo ocular e pálpebra | deflexões amplas e lentas, máximas na região frontopolar |
| Movimentos oculares | movimentos horizontais e verticais do olhar | variações em degrau nos elétrodos periorbitários |
| Atividade muscular | músculos temporais, masséteres, cervicais | atividade rápida, de alta frequência, sobretudo temporal |
| Atividade cardíaca | atividade elétrica do coração (ECG) | pequenas deflexões periódicas, sincronizadas com o pulso |
| Interferência da rede elétrica | instalação elétrica (50 Hz na Europa, 60 Hz nos EUA) | oscilação sinusoidal constante, de frequência fixa |
| Sudação e movimento | pele e cabos | oscilações lentas da linha de base |
| Mau contacto do elétrodo | impedância elevada | sinal plano ou com ruído excessivo |

## 3. Filtragem

O sinal de EEG resulta da sobreposição de oscilações de diferentes **frequências**, medidas em **Hertz (Hz)**: uma oscilação de 10 Hz completa 10 ciclos por segundo. Os ritmos clínicos clássicos (delta, teta, alfa, beta) situam-se aproximadamente entre 1 e 30 Hz.

A **densidade espectral de potência** indica a contribuição de cada frequência para o sinal registado:

In [ ]:
spectrum(raw, "Densidade espectral de potência: sinal não processado")

A atividade cerebral de interesse situa-se, na maioria das aplicações, **entre 1 e 40 Hz**:
- **abaixo de 1 Hz** predominam oscilações lentas da linha de base (sudação, movimento dos cabos);
- **acima de 40 Hz** predominam a atividade muscular e o ruído elétrico;
- observa-se um **pico estreito nos 60 Hz**, que corresponde à **interferência da rede elétrica**. Estes dados foram adquiridos nos EUA, onde a rede opera a 60 Hz; em Portugal e no resto da Europa, a interferência surgiria nos 50 Hz.

Aplica-se um **filtro passa-banda de 1 a 40 Hz**, que atenua as frequências fora deste intervalo. O filtro é aplicado a uma cópia (`raw.copy()`), para manter o sinal original disponível para comparação.

In [ ]:
filtered = raw.copy().filter(l_freq=1.0, h_freq=40.0)

spectrum(filtered, "Densidade espectral de potência: após filtragem (1-40 Hz)")

As frequências acima de 40 Hz, incluindo a interferência nos 60 Hz, foram atenuadas. Comparação do sinal antes e depois da filtragem:

In [ ]:
show(raw, "Antes da filtragem")
show(filtered, "Após filtragem (1-40 Hz)")

O sinal filtrado apresenta uma linha de base mais estável e menos ruído de alta frequência. **No entanto, os pestanejos mantêm-se.**

Isto deve-se ao facto de o conteúdo espectral do pestanejo (predominantemente abaixo de 5 Hz) se sobrepor ao da atividade cerebral. A filtragem não permite, por isso, separá-los, sendo necessário recorrer a um método de separação espacial (secção 6).

## 4. Elétrodos com mau contacto

Um elétrodo pode perder o contacto adequado com o couro cabeludo (gel seco, cabelo interposto, cabo solto), passando a registar sobretudo ruído.

Como este registo não apresenta esse problema, o mesmo será **simulado**, adicionando ruído aleatório (desvio-padrão de 60 µV) ao elétrodo **Cz** (vértex):

In [ ]:
broken = filtered.copy()
noise = np.random.default_rng(0).normal(0, 60e-6, broken.n_times)
broken._data[broken.ch_names.index("Cz")] += noise

show(broken, "Simulação de mau contacto no elétrodo Cz")

O procedimento habitual consiste em:

1. **marcar o elétrodo como inválido**;
2. **interpolá-lo**, isto é, estimar o sinal que deveria ter sido registado a partir dos **elétrodos vizinhos** (interpolação esférica).

In [ ]:
broken.info["bads"] = ["Cz"]                   # 1. marcar o elétrodo como inválido
repaired = broken.copy().interpolate_bads()    # 2. interpolá-lo a partir dos elétrodos vizinhos

show(repaired, "Elétrodo Cz após interpolação")

O sinal interpolado no elétrodo Cz é consistente com o dos elétrodos adjacentes.

A análise prossegue com o registo filtrado original (`filtered`), que não contém esta simulação.

## 5. O artefacto ocular

O MNE permite **detetar automaticamente todos os pestanejos**, identificando as deflexões de grande amplitude nos elétrodos frontopolares, e extrair um segmento de 1 segundo centrado em cada um. A **média desses segmentos** corresponde ao perfil típico do artefacto.

(Este registo não inclui um canal de eletro-oculografia (EOG) dedicado; por isso, são utilizados os elétrodos **Fp1** e **Fp2**, os mais próximos dos olhos.)

In [ ]:
blink_epochs = mne.preprocessing.create_eog_epochs(filtered, ch_name=["Fp1", "Fp2"], baseline=(-0.5, -0.2))
print(f"Número de pestanejos detetados: {len(blink_epochs)}")

average_blink = blink_epochs.average()
show_average(average_blink, "Pestanejo médio", topomap=True)

Interpretação da figura:
- **as curvas** correspondem aos elétrodos; o instante **0** corresponde ao pico do pestanejo;
- **o mapa topográfico** representa a distribuição da amplitude no escalpe nesse instante. O artefacto é **máximo na região frontopolar** e decresce no sentido posterior.

**É esta distribuição espacial característica que permite separar o artefacto da atividade cerebral**, conforme descrito na secção seguinte.

## 6. Análise de Componentes Independentes (ICA)

Considere-se uma sala onde **várias pessoas falam em simultâneo** e onde estão colocados **vários microfones**. Cada microfone regista uma **mistura** de todas as vozes, com maior contribuição das pessoas mais próximas. Este cenário é conhecido como o **problema do *cocktail party***.

A **Análise de Componentes Independentes** (*Independent Component Analysis*, ICA) é um método estatístico que permite, a partir dessas gravações, **estimar separadamente cada uma das fontes**.

O EEG constitui um problema equivalente:
- as **fontes** são as diferentes origens de atividade elétrica: regiões corticais, mas também os **olhos**, os **músculos** e o **coração**;
- os **microfones** são os 64 elétrodos, cada um a registar uma mistura de todas as fontes.

O procedimento compreende três passos:
1. **decompor** o sinal em fontes, designadas **componentes**;
2. **identificar o componente** correspondente ao pestanejo;
3. **excluí-lo** e reconstruir o sinal a partir dos restantes componentes.

```
 elétrodos (sinal misto)      componentes (fontes estimadas)        elétrodos (sinal corrigido)
  Fp1  ~~~^~~~^~~               cerebral   ~~~~~~~~~                   Fp1  ~~~~~~~~~
  Fz   ~~~^~~~^~~  ── ICA ──▶   cerebral   ~~~~~~~~~  ── exclusão ──▶  Fz   ~~~~~~~~~
  Oz   ~~~~~~~~~~               pestanejo  ___^___^_                   Oz   ~~~~~~~~~
```

Decomposição do sinal filtrado em 20 componentes (o parâmetro `random_state` torna o resultado reprodutível):

In [ ]:
ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter="auto")
ica.fit(filtered)

A figura seguinte apresenta a **distribuição topográfica** de cada componente.

**Qual destes componentes corresponde ao pestanejo?** Recorde-se a topografia do pestanejo médio da secção 5: máxima na região frontopolar.

In [ ]:
ica.plot_components(sphere="eeglab");

Para confirmar, o MNE calcula a **correlação** entre cada componente e os elétrodos frontopolares (valores próximos de 1 ou -1 indicam elevada semelhança com o pestanejo; valores próximos de 0 indicam ausência de relação). É apresentado um gráfico por elétrodo (Fp1 em cima, Fp2 em baixo); o componente identificado surge **a vermelho**.

In [ ]:
blink_components, scores = ica.find_bads_eog(filtered, ch_name=["Fp1", "Fp2"])
print("Componente(s) identificado(s) como pestanejo:", [int(c) for c in blink_components])

show_correlation(ica, scores, blink_components)

A **evolução temporal** dos primeiros componentes permite uma confirmação adicional: o componente associado ao pestanejo deve apresentar uma deflexão em cada pestanejo, enquanto os restantes apresentam atividade contínua.

In [ ]:
show_components_over_time(ica, filtered)

O componente **ICA000** apresenta uma deflexão marcada em cada pestanejo, ao contrário dos restantes. A polaridade (aqui negativa) não tem significado: na ICA, o sinal de cada componente é arbitrário.

## 7. Avaliação do resultado

O componente do pestanejo é excluído e o sinal é reconstruído a partir dos restantes componentes.

In [ ]:
ica.exclude = blink_components
cleaned = ica.apply(filtered.copy())

show(filtered, "Antes da correção (sinal filtrado)", mark_blinks=True)
show(cleaned, "Após exclusão do componente ocular", mark_blinks=True)

Os pestanejos deixaram de ser visíveis em Fp1 e Fp2, e **a restante atividade foi preservada**.

Sobreposição do elétrodo **Fp1** antes (vermelho) e depois (preto) da correção:

In [ ]:
compare(filtered, cleaned, "Elétrodo Fp1: antes e depois da correção")

Por fim, compara-se o **pestanejo médio** antes e depois da correção, com a mesma escala, nos mesmos instantes.

In [ ]:
cleaned_blink = mne.Epochs(cleaned, blink_epochs.events, tmin=-0.5, tmax=0.5, baseline=(-0.5, -0.2)).average()

show_average(average_blink, "Pestanejo médio: antes da correção", ylim=dict(eeg=[-150, 250]))
show_average(cleaned_blink, "Mesmos instantes: após a correção", ylim=dict(eeg=[-150, 250]))

Nos instantes correspondentes aos pestanejos, a amplitude residual é **mínima**, o que confirma a eficácia da correção.

## Síntese

| Problema | Método | Instrução |
|---|---|---|
| Oscilações lentas e ruído de alta frequência | **Filtragem passa-banda** | `raw.copy().filter(l_freq=1.0, h_freq=40.0)` |
| Elétrodo com mau contacto | **Interpolação** | `info["bads"] = [...]`, `interpolate_bads()` |
| Artefacto ocular | **ICA** | `ica.fit(...)`, `ica.exclude = [...]`, `ica.apply(...)` |

O pré-processamento constitui normalmente a **primeira etapa** de qualquer estudo de EEG. Só após esta etapa se procede à análise da atividade cerebral propriamente dita.

## Exercícios

1. Na secção 1, substitua o nome do ficheiro por outro registo da mesma base de dados (por exemplo, `"S003R04.edf"` ou `"S007R04.edf"`) e execute novamente todo o notebook (**Runtime → Run all**). O artefacto ocular apresenta as mesmas características em todos os participantes?
2. Na secção 7, exclua um componente **diferente**, por exemplo `ica.exclude = [5]`. Que alterações se observam no sinal?
3. Na secção 4, simule o mau contacto noutro elétrodo, como `"Oz"` ou `"Fp1"`.
4. Na secção 3, altere os limites do filtro (por exemplo, `h_freq=20.0`) e observe o efeito no sinal.

---
### Nota: utilização de registos próprios

Para aplicar este processamento a registos próprios, de forma automática, utilize o notebook `02_pipeline_completo.ipynb`.